# Aula 5, TF-IDF

Notebook executável que acompanha a aula [05-tf-idf.md](../../lessons/modulo-03-fundamentos-nlp/05-tf-idf.md).

## O que vamos fazer aqui

O TF-IDF refina o Bag of Words dando a cada palavra um peso que combina o quanto ela é
frequente em um documento e o quanto é rara na coleção. Vamos calcular esses pesos do
zero e, com eles, construir o classificador de perguntas por tema que fecha o módulo.
Python puro.

## TF-IDF do zero

Primeiro calculamos o IDF de cada palavra, que é o logaritmo da razão entre o total de
documentos e quantos contêm a palavra. Depois, o peso TF-IDF de uma palavra em um
texto é a sua contagem multiplicada pelo IDF. Vamos inspecionar os pesos de uma
pergunta.

In [ ]:
import re
import math
from collections import Counter

treino = [
    ("como faço a derivada de uma função", "cálculo"),
    ("qual a regra da cadeia na derivada", "cálculo"),
    ("o que é a integral de uma função", "cálculo"),
    ("como resolvo um sistema linear com matrizes", "álgebra"),
    ("o que é um autovetor de uma matriz", "álgebra"),
    ("como multiplico duas matrizes", "álgebra"),
    ("como declaro uma função em python", "programação"),
    ("o que é um laço de repetição em python", "programação"),
    ("como uso uma lista em python", "programação"),
]
documentos = [texto for texto, _ in treino]


def tokenizar(texto):
    return re.findall(r"\w+", texto.lower(), re.UNICODE)


N = len(documentos)
df = Counter()
for doc in documentos:
    for palavra in set(tokenizar(doc)):
        df[palavra] += 1
idf = {palavra: math.log(N / freq) for palavra, freq in df.items()}


def vetor_tfidf(texto):
    tf = Counter(tokenizar(texto))
    return {palavra: tf[palavra] * idf.get(palavra, 0.0) for palavra in tf}


# Mostra as palavras de maior peso em uma pergunta.
exemplo = vetor_tfidf("como faço a derivada de uma função")
print("Pesos TF-IDF (maiores primeiro):")
for palavra, peso in sorted(exemplo.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {palavra:10} {peso:.3f}")

Repare que palavras específicas, como derivada e faço, recebem peso alto,
enquanto palavras espalhadas por muitos documentos, como como e função, recebem peso
baixo ou até zero. É essa ponderação que dá ao TF-IDF o seu poder de discriminar
temas, algo que faltava ao Bag of Words.

## Classificador de tema por similaridade

Para classificar, representamos cada tema por um centróide, a soma dos vetores TF-IDF
das suas perguntas. Uma pergunta nova recebe o tema cujo centróide é mais parecido com
ela, pela similaridade do cosseno.

In [ ]:
def cosseno(a, b):
    produto = sum(a[p] * b.get(p, 0.0) for p in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb_ = math.sqrt(sum(v * v for v in b.values()))
    return produto / (na * nb_) if na and nb_ else 0.0


temas = {}
for texto, tema in treino:
    v = vetor_tfidf(texto)
    alvo = temas.setdefault(tema, {})
    for palavra, peso in v.items():
        alvo[palavra] = alvo.get(palavra, 0.0) + peso


def classificar(texto):
    v = vetor_tfidf(texto)
    return max(temas, key=lambda tema: cosseno(v, temas[tema]))


teste = [
    ("como calculo a derivada de x", "cálculo"),
    ("como inverto uma matriz", "álgebra"),
    ("como crio uma função em python", "programação"),
]
acertos = 0
for pergunta, correto in teste:
    previsto = classificar(pergunta)
    acertos += (previsto == correto)
    print(f"{pergunta!r:42} -> {previsto:12} (correto: {correto})")
print(f"\nAcurácia no teste: {acertos}/{len(teste)}")

As três perguntas novas caem no tema certo, mesmo usando palavras que não
estão idênticas no treino, porque o TF-IDF dá peso aos termos certos. Esse é o
pagamento de tudo o que construímos no módulo, um classificador de texto simples,
transparente e que funciona.

Para o projeto do módulo, monte um conjunto maior de perguntas rotuladas, separe
algumas para teste, meça a acurácia e comente um acerto, um erro e o que o TF-IDF
acrescentou em relação à contagem crua do Bag of Words.